In [0]:
USE CATALOG smart_odoo;

CREATE OR REPLACE VIEW gold.vw_inventory_kpis AS
WITH base AS
(
    SELECT
        product_id,
        stock_value,
        available_quantity,
        is_low_stock,
        location_type,
        snapshot_date,
        inventory_date
    FROM gold.fact_inventory
    WHERE quantity > 0
),
current_month_value AS
(
    SELECT SUM(stock_value) AS value
    FROM base
    WHERE snapshot_date >= DATE_TRUNC('month', current_date())
),
last_month_value AS
(
    SELECT SUM(stock_value) AS value
    FROM base
    WHERE snapshot_date >= DATE_TRUNC('month', ADD_MONTHS(current_date(), -1))
      AND snapshot_date <  DATE_TRUNC('month', current_date())
)
SELECT
    -- KPI 1: Total Items
    COUNT(DISTINCT b.product_id)                            AS total_items,
 
    COUNT(DISTINCT CASE
        WHEN CAST(b.inventory_date AS DATE)
             >= DATE_TRUNC('month', current_date())
        THEN b.product_id
    END)                                                    AS items_added_this_month,
 
    -- KPI 2: Total Value
    ROUND(MAX(cm.value), 2)                                 AS total_inventory_value,
 
    ifnull(ROUND(
        CASE
            WHEN MAX(lm.value) > 0
            THEN ((MAX(cm.value) - MAX(lm.value)) / MAX(lm.value)) * 100
            ELSE NULL
        END, 2
    ),0)                                                       AS value_pct_change_from_last_month,
 
    -- KPI 3: Low Stock
    COUNT(DISTINCT CASE
        WHEN b.is_low_stock = TRUE THEN b.product_id
    END)                                                    AS low_stock_items,
 
    -- KPI 4: Categories (distinct active location types)
    COUNT(DISTINCT b.location_type)                         AS active_categories
 
FROM base b
CROSS JOIN current_month_value cm
CROSS JOIN last_month_value lm;


In [0]:
CREATE OR REPLACE VIEW gold.vw_stock_movement_trend AS
SELECT
    dd.month_short                              AS month_name,    -- e.g. Jan, Feb, Aug
    dd.year_number                              AS year_number,
    dd.month_number                             AS month_number,  -- for correct ordering
 
    SUM(CASE WHEN sm.is_in  = TRUE THEN sm.quantity ELSE 0 END)  AS inbound_qty,
    SUM(CASE WHEN sm.is_out = TRUE THEN sm.quantity ELSE 0 END)  AS outbound_qty,
 
    -- Net difference: positive = more came in, negative = more went out
    SUM(CASE WHEN sm.is_in  = TRUE THEN sm.quantity ELSE 0 END)
  - SUM(CASE WHEN sm.is_out = TRUE THEN sm.quantity ELSE 0 END)  AS net_movement,
 
    SUM(CASE WHEN sm.is_in  = TRUE THEN sm.value ELSE 0 END)     AS inbound_value,
    SUM(CASE WHEN sm.is_out = TRUE THEN sm.value ELSE 0 END)     AS outbound_value,
 
    SUM(CASE WHEN sm.is_in  = TRUE THEN sm.value ELSE 0 END)
  - SUM(CASE WHEN sm.is_out = TRUE THEN sm.value ELSE 0 END)     AS net_value
 
FROM silver.stock_move sm
 
-- join dim_date on the move date
INNER JOIN gold.dim_date dd
    ON dd.full_date = CAST(sm.date AS DATE)
 
WHERE sm.state = 'done'
  AND dd.year_number = YEAR(current_date())
 
GROUP BY
    dd.month_short,
    dd.year_number,
    dd.month_number
 
ORDER BY
    dd.month_number ASC;


In [0]:
CREATE OR REPLACE VIEW gold.vw_inventory_by_category AS
SELECT
    location_type                           AS category,
    COUNT(DISTINCT product_id)              AS total_products,
    ROUND(SUM(quantity), 2)                 AS total_quantity,
    ROUND(SUM(available_quantity), 2)       AS total_available_quantity,
    ROUND(SUM(stock_value), 2)              AS total_stock_value,
    COUNT(DISTINCT location_id)             AS total_locations
 
FROM gold.fact_inventory
WHERE quantity > 0
  AND location_type IS NOT NULL
 
GROUP BY location_type
 
ORDER BY total_stock_value DESC
 
LIMIT 5;


In [0]:
USE CATALOG smart_odoo;

CREATE OR REPLACE VIEW gold.vw_inventory_list AS
SELECT
    fi.product_id,
    dp.product_name,
    dp.product_sku,
    dp.categ_name                               AS category,

    ROUND(SUM(fi.quantity), 2)                 AS quantity_on_hand,
    ROUND(SUM(fi.available_quantity), 2)       AS available_quantity,
    ROUND(MAX(dp.product_list_price), 2)        AS unit_price,
    ROUND(MAX(dp.product_cost), 2)              AS unit_cost,
    ROUND(SUM(fi.stock_value), 2)              AS total_stock_value,
 
    CASE
        WHEN SUM(fi.available_quantity) <= 0                       THEN 'Out of Stock'
        WHEN SUM(fi.available_quantity) < MAX(fi.product_min_qty) THEN 'Low Stock'
        ELSE                                                              'In Stock'
    END                                         AS status
 
FROM gold.fact_inventory fi
INNER JOIN gold.dim_product dp
    ON dp.product_id = CAST(fi.product_id AS INT)
 
GROUP BY
    fi.product_id,
    dp.product_name,
    dp.product_sku,
    dp.categ_name

ORDER BY product_id ASC;


In [0]:
Select * from smart_odoo.gold.vw_inventory_kpis;
---------------------------------------------------
Select * from smart_odoo.gold.vw_stock_movement_trend;
---------------------------------------------------
Select * from smart_odoo.gold.vw_inventory_by_category;
---------------------------------------------------
Select * from smart_odoo.gold.vw_inventory_list;